# Pandas：从公司财务表到可分析数据

Pandas 是 Python 中处理表格数据的核心工具。它提供了类似电子表格的数据结构，又可以用代码完成重复、精确、可追踪的数据整理和计算。在经管数据分析中，上市公司财务表、问卷数据、交易数据、地区统计数据等，通常都可以先整理成 pandas 的 `DataFrame`，再继续做统计分析、建模或可视化。


**Jupyter Notebook 简介**

本章使用 Jupyter Notebook。Notebook 文件扩展名是 `.ipynb`，适合记录数据分析过程：代码、文字说明和运行结果可以放在同一个文件中。

**数据存放约定**

本课程约定：数据文件都放在工作目录下的 `data` 文件夹中。工作目录可以理解为当前项目或当前 notebook 所在的主要目录。读取数据时，可以用相对路径引用文件，例如 `data/finance_teaching_clean.xlsx`。

**本章使用的数据**

本章使用两张教学表：

- `data/finance_teaching_clean.xlsx`：公司年度财务数据。
- `data/company_profile_teaching_clean.xlsx`：公司基本信息。

## 本章知识点安排

**阶段 1：从一张财务表得到 2020 年公司表现排名**

读取 Excel；查看 `head()`、`tail()`、`shape`、`columns`、`dtypes`、`info()`、`describe()`；理解 `Series` 和 `DataFrame`；选列；`loc` / `iloc`；条件筛选和复合条件；`query()`；列运算；`pd.cut()`；按条件赋值；排序；简单统计。

**阶段 2：处理证券代码，并合并公司信息**

编号列的字符串处理；`converters`；`str.zfill()`；`str.strip()`；`str.contains()`；`pd.to_datetime()`；`.dt.year`、`.dt.month`、`.dt.quarter`；`value_counts()`；`rename()`；`merge()`；`left_on` / `right_on`；合并后检查。

**阶段 3：处理真实数据中的常见小问题**

构造工作副本；检查缺失值；检查重复行；`drop_duplicates()`；清理特殊文本；`pd.to_numeric()`；`dropna()`；`fillna()`；`replace()`；`map()`。

**阶段 4：按行业和年份做汇总分析**

`groupby()`；多指标 `agg()`；自定义聚合函数；按多列分组；分组后排序；每组取前几名；分组循环；`concat()`。

**阶段 5：重建更适合分析的数据表**

`pivot_table()`；长表和宽表；构造公司层面摘要表；`set_index()`；`reset_index()`；构造 `Series` / `DataFrame`；`to_excel()`；`to_csv()`；`read_csv()`。

**阶段 6：时间序列入门**

`date_range()`；日期索引；按日期切片；`shift()`；`diff()`；`pct_change()`；`cumprod()`；`resample()`。

**开始前：DataFrame 和 Series**

Pandas 主要处理二维表格。一个 `DataFrame` 可以理解为一张 Excel 表：有行、列、列名和行索引。一个 `Series` 可以理解为一列数据，它由索引和值组成。多个 `Series` 横向放在一起，就形成一个 `DataFrame`。

```text
index + ndarray -> Series
index + Series + Series + ... -> DataFrame
```

![](images/df-dp.png)

**Excel 和 CSV**

常见表格数据主要有两类：

- Excel 文件：扩展名通常是 `.xlsx`。可以保存格式、颜色、多个工作表等信息，适合给人查看和编辑。
- CSV 文件：扩展名是 `.csv`。本质上是纯文本，只保存数据本身，不保存格式。它体积小、通用性强，几乎所有数据软件都能读取。

在数据分析中，Excel 和 CSV 都常见。拿不准保存成什么格式时，CSV 往往更通用；需要给人直接打开查看时，Excel 更方便。

## 阶段 1：从一张财务表得到 2020 年公司表现排名

目标：读入年度财务表，选出 2020 年公司，计算几个财务指标，并做出一张公司排名表。

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.4f}".format)

finance_raw = pd.read_excel("data/finance_teaching_clean.xlsx")
finance_raw.head()

,证券代码,证券简称,年份,总资产_亿元,营业收入_亿元,净利润_亿元,资产负债率
0,1,平安银行,2018,34185.9200,1062.1200,248.1800,0.9298
1,1,平安银行,2019,39390.7000,1268.1400,281.9500,0.9205
2,1,平安银行,2020,44685.1400,1432.4200,289.2800,0.9185
3,2,万科A,2018,15285.7900,2976.7900,492.7200,0.8459
4,2,万科A,2019,17299.2900,3678.9400,551.3200,0.8436


先查看数据规模、字段和类型。真实工作中，先确认表长什么样，后面的计算会更稳妥。

In [2]:
print("行列数：", finance_raw.shape)
print("列名：", finance_raw.columns.tolist())
print("数据类型：")
print(finance_raw.dtypes)

finance_raw.info()

行列数： (84, 7)
列名： ['证券代码', '证券简称', '年份', '总资产_亿元', '营业收入_亿元', '净利润_亿元', '资产负债率']
数据类型：
证券代码         int64
证券简称        object
年份           int64
总资产_亿元     float64
营业收入_亿元    float64
净利润_亿元     float64
资产负债率      float64
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   证券代码     84 non-null     int64  
 1   证券简称     84 non-null     object 
 2   年份       84 non-null     int64  
 3   总资产_亿元   84 non-null     float64
 4   营业收入_亿元  84 non-null     float64
 5   净利润_亿元   84 non-null     float64
 6   资产负债率    84 non-null     float64
dtypes: float64(4), int64(2), object(1)
memory usage: 4.7+ KB


In [3]:
finance_raw.tail()

,证券代码,证券简称,年份,总资产_亿元,营业收入_亿元,净利润_亿元,资产负债率
79,600015,华夏银行,2019,30207.8900,827.6900,221.1500,0.9108
80,600015,华夏银行,2020,33998.1600,927.1700,215.6800,0.9169
81,600016,民生银行,2018,59948.2200,1288.9300,503.3000,0.9281
82,600016,民生银行,2019,66818.4100,1549.8200,549.2400,0.9206
83,600016,民生银行,2020,69502.3300,1667.7700,351.0200,0.9221


In [4]:
finance_raw.describe()

,证券代码,年份,总资产_亿元,营业收入_亿元,净利润_亿元,资产负债率
count,84.0000,84.0000,84.0000,84.0000,84.0000,84.0000
mean,64485.4286,2019.0000,8263.2971,560.8667,91.8493,0.5904
std,186625.5943,0.8214,19255.0316,915.6682,173.1629,0.2229
min,1.0000,2018.0000,3.5100,0.4200,-4.4500,0.0602
25%,19.0000,2018.0000,84.8350,33.0025,0.6025,0.4723
50%,155.5000,2019.0000,146.7800,107.8400,5.7900,0.6060
75%,541.0000,2020.0000,927.3325,586.9000,69.5200,0.7074
max,600016.0000,2020.0000,79502.1800,4191.1200,595.0600,0.9298


`证券代码` 是编号，主要用于识别和合并公司。阶段 1 先集中处理财务指标，因此把它临时放到一边。这里使用 `.copy()` 明确生成工作副本。

In [5]:
finance = finance_raw.drop(columns=["证券代码"]).copy()
finance.head()

,证券简称,年份,总资产_亿元,营业收入_亿元,净利润_亿元,资产负债率
0,平安银行,2018,34185.9200,1062.1200,248.1800,0.9298
1,平安银行,2019,39390.7000,1268.1400,281.9500,0.9205
2,平安银行,2020,44685.1400,1432.4200,289.2800,0.9185
3,万科A,2018,15285.7900,2976.7900,492.7200,0.8459
4,万科A,2019,17299.2900,3678.9400,551.3200,0.8436


注意：pandas 中很多“修改”操作会返回一张新表。例如 `drop()` 会返回删除列之后的新表。如果希望后续继续使用结果，通常要把结果赋值给一个变量。

```python
finance = finance_raw.drop(columns=["证券代码"]).copy()
```

初学时把每一步结果明确赋值给变量，通常更清楚，也更容易检查。和依赖 `inplace=True` 相比，这种写法更便于追踪分析过程。

一列数据是 `Series`，多列数据是 `DataFrame`。这是 pandas 最基本的两个对象。

In [6]:
one_col = finance["营业收入_亿元"]
some_cols = finance[["证券简称", "年份", "营业收入_亿元"]]

print(type(one_col))
print(type(some_cols))

<class 'pandas.core.series.Series'>
<class 'pandas.core.frame.DataFrame'>


选择列时，单个列名返回 `Series`；列名列表返回 `DataFrame`。

In [7]:
finance[["证券简称", "年份", "营业收入_亿元", "净利润_亿元"]].head()

,证券简称,年份,营业收入_亿元,净利润_亿元
0,平安银行,2018,1062.1200,248.1800
1,平安银行,2019,1268.1400,281.9500
2,平安银行,2020,1432.4200,289.2800
3,万科A,2018,2976.7900,492.7200
4,万科A,2019,3678.9400,551.3200


`.iloc` 按位置选择，适合快速查看“第几行、第几列”。

In [8]:
finance.iloc[:5, :4]

,证券简称,年份,总资产_亿元,营业收入_亿元
0,平安银行,2018,34185.9200,1062.1200
1,平安银行,2019,39390.7000,1268.1400
2,平安银行,2020,44685.1400,1432.4200
3,万科A,2018,15285.7900,2976.7900
4,万科A,2019,17299.2900,3678.9400


`.loc` 按标签选择，常和条件一起使用。

In [9]:
finance.loc[
    finance["年份"] == 2020,
    ["证券简称", "营业收入_亿元", "净利润_亿元", "资产负债率"],
].head()

,证券简称,营业收入_亿元,净利润_亿元,资产负债率
2,平安银行,1432.4200,289.2800,0.9185
5,万科A,4191.1200,592.9800,0.8128
8,国华网安,2.8100,0.6200,0.0683
11,深振业A,29.3500,9.0300,0.4922
14,*ST 全新,0.4500,-1.2300,0.8139


`.loc` 也可以用标签切片。要注意：`.loc` 的标签切片包含结束点，这和 Python 列表切片不同。

In [10]:
finance.loc[0:3, "证券简称":"营业收入_亿元"]

,证券简称,年份,总资产_亿元,营业收入_亿元
0,平安银行,2018,34185.9200,1062.1200
1,平安银行,2019,39390.7000,1268.1400
2,平安银行,2020,44685.1400,1432.4200
3,万科A,2018,15285.7900,2976.7900


复合条件要用 `&`、`|`、`~`，每个条件外面加括号。

In [11]:
good_2020 = finance.loc[
    (finance["年份"] == 2020)
    & (finance["净利润_亿元"] > 0)
    & (finance["营业收入_亿元"].notna())
    & (finance["资产负债率"] < 0.7),
    ["证券简称", "营业收入_亿元", "净利润_亿元", "资产负债率"],
]

good_2020.head()

,证券简称,营业收入_亿元,净利润_亿元,资产负债率
8,国华网安,2.8100,0.6200,0.0683
11,深振业A,29.3500,9.0300,0.4922
17,深物业A,41.0400,7.3100,0.6903
23,深华发A,6.9200,0.0700,0.4644
26,深科技,149.6700,9.5300,0.6298


筛选之后最好看一眼结果。`head()` 看前几行，`tail()` 看后几行，能帮助我们发现结果是否大致符合预期。

In [12]:
print("筛选结果行数：", len(good_2020))
display(good_2020.head())
display(good_2020.tail())

筛选结果行数： 18


,证券简称,营业收入_亿元,净利润_亿元,资产负债率
8,国华网安,2.8100,0.6200,0.0683
11,深振业A,29.3500,9.0300,0.4922
17,深物业A,41.0400,7.3100,0.6903
23,深华发A,6.9200,0.0700,0.4644
26,深科技,149.6700,9.5300,0.6298


,证券简称,营业收入_亿元,净利润_亿元,资产负债率
56,丽珠集团,105.2000,21.3100,0.3376
62,云南白药,327.4300,55.1100,0.3056
65,江铃汽车,330.9600,5.5100,0.6102
68,神州信息,106.8600,4.6600,0.5282
71,万向钱潮,108.8200,4.4400,0.4290


`query()` 可以把筛选条件写成字符串。条件较短时，这种写法很方便。

In [13]:
finance.query("年份 == 2020 and 净利润_亿元 > 0").head()

,证券简称,年份,总资产_亿元,营业收入_亿元,净利润_亿元,资产负债率
2,平安银行,2020,44685.1400,1432.4200,289.2800,0.9185
5,万科A,2020,18691.7700,4191.1200,592.9800,0.8128
8,国华网安,2020,15.6400,2.8100,0.6200,0.0683
11,深振业A,2020,154.3500,29.3500,9.0300,0.4922
17,深物业A,2020,122.0700,41.0400,7.3100,0.6903


用已有列生成新列，是 pandas 中最常见的操作之一。

In [14]:
finance["净利率"] = finance["净利润_亿元"] / finance["营业收入_亿元"]
finance["资产收益率"] = finance["净利润_亿元"] / finance["总资产_亿元"]
finance["是否盈利"] = finance["净利润_亿元"] > 0
finance["资产规模"] = pd.cut(
    finance["总资产_亿元"],
    bins=[0, 100, 1000, np.inf],
    labels=["小", "中", "大"],
)

finance[["证券简称", "年份", "净利率", "资产收益率", "是否盈利", "资产规模"]].head()

,证券简称,年份,净利率,资产收益率,是否盈利,资产规模
0,平安银行,2018,0.2337,0.0073,True,大
1,平安银行,2019,0.2223,0.0072,True,大
2,平安银行,2020,0.2020,0.0065,True,大
3,万科A,2018,0.1655,0.0322,True,大
4,万科A,2019,0.1499,0.0319,True,大


也可以按条件给新列赋值。

In [15]:
finance["负债水平"] = "正常"
finance.loc[finance["资产负债率"] >= 0.7, "负债水平"] = "较高"

finance[["证券简称", "年份", "资产负债率", "负债水平"]].head()

,证券简称,年份,资产负债率,负债水平
0,平安银行,2018,0.9298,较高
1,平安银行,2019,0.9205,较高
2,平安银行,2020,0.9185,较高
3,万科A,2018,0.8459,较高
4,万科A,2019,0.8436,较高


按一个或多个指标排序，得到阶段性结果：2020 年公司表现排名。

In [16]:
finance_2020_rank = (
    finance[finance["年份"] == 2020]
    .sort_values(["营业收入_亿元", "净利率"], ascending=[False, False])
)

finance_2020_rank[[
    "证券简称", "营业收入_亿元", "净利润_亿元", "净利率", "资产收益率", "资产负债率", "负债水平"
]].head(10)

,证券简称,营业收入_亿元,净利润_亿元,净利率,资产收益率,资产负债率,负债水平
5,万科A,4191.1200,592.9800,0.1415,0.0317,0.8128,较高
47,美的集团,2857.1000,275.0700,0.0963,0.0763,0.6553,正常
50,潍柴动力,1974.9100,112.7500,0.0571,0.0416,0.7029,较高
77,浦发银行,1746.8700,589.9300,0.3377,0.0074,0.9188,较高
83,民生银行,1667.7700,351.0200,0.2105,0.0051,0.9221,较高
2,平安银行,1432.4200,289.2800,0.2020,0.0065,0.9185,较高
80,华夏银行,927.1700,215.6800,0.2326,0.0063,0.9169,较高
20,深康佳A,503.5200,5.4000,0.0107,0.0108,0.7851,较高
65,江铃汽车,330.9600,5.5100,0.0166,0.0195,0.6102,正常
62,云南白药,327.4300,55.1100,0.1683,0.0998,0.3056,正常


排序后也要看一眼尾部。前几行告诉我们谁排在前面，后几行能帮助我们理解这个排名的另一端。

In [17]:
finance_2020_rank[[
    "证券简称", "营业收入_亿元", "净利润_亿元", "净利率", "资产负债率"
]].tail(5)

,证券简称,营业收入_亿元,净利润_亿元,净利率,资产负债率
32,深纺织A,21.0900,0.4300,0.0204,0.2154
23,深华发A,6.9200,0.0700,0.0101,0.4644
8,国华网安,2.8100,0.6200,0.2206,0.0683
74,*ST 万方,1.1100,-0.2000,-0.1802,0.6681
14,*ST 全新,0.4500,-1.2300,-2.7333,0.8139


简单统计可以快速了解样本。

In [18]:
finance_2020 = finance[finance["年份"] == 2020]

print("公司数量：", finance_2020["证券简称"].nunique())
print("营业收入平均值：", finance_2020["营业收入_亿元"].mean())
print("营业收入最大值：", finance_2020["营业收入_亿元"].max())

finance_2020[["营业收入_亿元", "净利润_亿元", "资产负债率", "净利率"]].describe()

公司数量： 28
营业收入平均值： 614.4621428571428
营业收入最大值： 4191.12


,营业收入_亿元,净利润_亿元,资产负债率,净利率
count,28.0000,28.0000,28.0000,28.0000
mean,614.4621,91.6954,0.5952,-0.0061
std,1025.9494,173.2019,0.2360,0.5461
min,0.4500,-1.2300,0.0683,-2.7333
25%,33.8700,0.6100,0.4556,0.0122
50%,109.9750,6.4100,0.6425,0.0503
75%,609.4325,69.5200,0.7234,0.1841
max,4191.1200,592.9800,0.9221,0.3377


为了展示结果，也可以重命名结果表中的列。

In [19]:
rank_display = finance_2020_rank.rename(
    columns={
        "营业收入_亿元": "营业收入",
        "净利润_亿元": "净利润",
    }
)

rank_display[["证券简称", "营业收入", "净利润", "净利率"]].head()

,证券简称,营业收入,净利润,净利率
5,万科A,4191.1200,592.9800,0.1415
47,美的集团,2857.1000,275.0700,0.0963
50,潍柴动力,1974.9100,112.7500,0.0571
77,浦发银行,1746.8700,589.9300,0.3377
83,民生银行,1667.7700,351.0200,0.2105


阶段 1 小结：我们得到了 `finance_2020_rank`。这一阶段带出了读取、查看、选行选列、`loc`、`iloc`、条件筛选、`query`、列运算、按条件赋值、排序和简单统计。

完成本阶段后，请做最后“练习”中的阶段 1 练习。

## 阶段 2：处理证券代码，并合并公司信息

目标：把财务表和公司信息表合并，让财务指标带上行业、省份、城市、上市日期等背景信息。

证券代码是编号，适合作为字符串处理。读取时把它转成字符串，并补齐到 6 位。

In [20]:
def read_code(x):
    return str(x).strip().zfill(6)

finance_code = pd.read_excel(
    "data/finance_teaching_clean.xlsx",
    converters={"证券代码": read_code},
)

company = pd.read_excel(
    "data/company_profile_teaching_clean.xlsx",
    converters={"证券代码": read_code},
)

finance_code.head()

,证券代码,证券简称,年份,总资产_亿元,营业收入_亿元,净利润_亿元,资产负债率
0,000001,平安银行,2018,34185.9200,1062.1200,248.1800,0.9298
1,000001,平安银行,2019,39390.7000,1268.1400,281.9500,0.9205
2,000001,平安银行,2020,44685.1400,1432.4200,289.2800,0.9185
3,000002,万科A,2018,15285.7900,2976.7900,492.7200,0.8459
4,000002,万科A,2019,17299.2900,3678.9400,551.3200,0.8436


公司信息表中有文本列和日期列，可以做几个轻量处理。

In [21]:
company = company.copy()
company["证券简称"] = company["证券简称"].str.strip()
company["行业名称"] = company["行业名称"].str.strip()
company["上市日期"] = pd.to_datetime(company["上市日期"])
company["上市年份"] = company["上市日期"].dt.year
company["是否ST"] = company["证券简称"].str.contains("ST", na=False)

company.head()

,证券代码,证券简称,行业名称,省份,城市,上市市场,上市日期,上市年份,是否ST
0,000001,平安银行,货币金融服务,广东省,深圳市,SZSE,1991-04-03,1991,False
1,000002,万科A,房地产业,广东省,深圳市,SZSE,1991-01-29,1991,False
2,000004,国华网安,软件和信息技术服务业,广东省,深圳市,SZSE,1991-01-14,1991,False
3,000006,深振业A,房地产业,广东省,深圳市,SZSE,1992-04-27,1992,False
4,000007,*ST 全新,房地产业,广东省,深圳市,SZSE,1992-04-13,1992,True


In [22]:
company["行业名称"].value_counts()

行业名称
货币金融服务              4
房地产业                4
软件和信息技术服务业          4
计算机、通信和其他电子设备制造业    4
汽车制造业               4
电气机械及器材制造业          4
医药制造业               4
Name: count, dtype: int64

字符串方法常用于筛选文本。比如找出行业名称中包含“制造”的公司，或找出简称中包含 `ST` 的公司。

In [23]:
manufacturing = company.loc[
    company["行业名称"].str.contains("制造", na=False),
    ["证券代码", "证券简称", "行业名称", "省份"],
]

print("制造业相关公司数量：", len(manufacturing))
display(manufacturing.head())
display(manufacturing.tail())

制造业相关公司数量： 16


,证券代码,证券简称,行业名称,省份
6,000016,深康佳A,计算机、通信和其他电子设备制造业,广东省
7,000020,深华发A,计算机、通信和其他电子设备制造业,广东省
8,000021,深科技,计算机、通信和其他电子设备制造业,广东省
9,000030,富奥股份,汽车制造业,吉林省
10,000045,深纺织A,计算机、通信和其他电子设备制造业,广东省


,证券代码,证券简称,行业名称,省份
18,000513,丽珠集团,医药制造业,广东省
19,000521,长虹美菱,电气机械及器材制造业,安徽省
20,000538,云南白药,医药制造业,云南省
21,000550,江铃汽车,汽车制造业,江西省
23,000559,万向钱潮,汽车制造业,浙江省


In [24]:
st_companies = company.loc[
    company["证券简称"].str.contains("ST", na=False),
    ["证券代码", "证券简称", "行业名称", "上市年份"],
]

st_companies

,证券代码,证券简称,行业名称,上市年份
4,000007,*ST 全新,房地产业,1992
24,000638,*ST 万方,软件和信息技术服务业,1996


日期方法常用于按时间筛选。比如根据上市日期提取年份、月份、季度，再筛选较早上市的公司。

In [25]:
company["上市月份"] = company["上市日期"].dt.month
company["上市季度"] = company["上市日期"].dt.quarter

old_listed = company.loc[
    company["上市年份"] < 2000,
    ["证券代码", "证券简称", "行业名称", "上市日期", "上市年份", "上市月份", "上市季度"],
].sort_values("上市日期")

display(old_listed.head())
display(old_listed.tail())

,证券代码,证券简称,行业名称,上市日期,上市年份,上市月份,上市季度
2,000004,国华网安,软件和信息技术服务业,1991-01-14,1991,1,1
1,000002,万科A,房地产业,1991-01-29,1991,1,1
0,000001,平安银行,货币金融服务,1991-04-03,1991,4,2
6,000016,深康佳A,计算机、通信和其他电子设备制造业,1992-03-27,1992,3,1
5,000011,深物业A,房地产业,1992-03-30,1992,3,1


,证券代码,证券简称,行业名称,上市日期,上市年份,上市月份,上市季度
10,000045,深纺织A,计算机、通信和其他电子设备制造业,1994-08-15,1994,8,3
11,000049,德赛电池,电气机械及器材制造业,1995-03-20,1995,3,1
17,000423,东阿阿胶,医药制造业,1996-07-29,1996,7,3
24,000638,*ST 万方,软件和信息技术服务业,1996-11-26,1996,11,4
25,600000,浦发银行,货币金融服务,1999-11-10,1999,11,4


重新在带证券代码的财务表中生成阶段 1 的指标。

In [26]:
finance_code["净利率"] = finance_code["净利润_亿元"] / finance_code["营业收入_亿元"]
finance_code["资产收益率"] = finance_code["净利润_亿元"] / finance_code["总资产_亿元"]
finance_code["是否盈利"] = finance_code["净利润_亿元"] > 0
finance_code["负债水平"] = np.where(finance_code["资产负债率"] >= 0.7, "较高", "正常")

finance_2020_rank_code = (
    finance_code[finance_code["年份"] == 2020]
    .sort_values("营业收入_亿元", ascending=False)
)

finance_2020_rank_code.head()

,证券代码,证券简称,年份,总资产_亿元,营业收入_亿元,净利润_亿元,资产负债率,净利率,资产收益率,是否盈利,负债水平
5,000002,万科A,2020,18691.7700,4191.1200,592.9800,0.8128,0.1415,0.0317,True,较高
47,000333,美的集团,2020,3603.8300,2857.1000,275.0700,0.6553,0.0963,0.0763,True,正常
50,000338,潍柴动力,2020,2707.5000,1974.9100,112.7500,0.7029,0.0571,0.0416,True,较高
77,600000,浦发银行,2020,79502.1800,1746.8700,589.9300,0.9188,0.3377,0.0074,True,较高
83,600016,民生银行,2020,69502.3300,1667.7700,351.0200,0.9221,0.2105,0.0051,True,较高


合并前把右表整理成需要的列。

In [27]:
company_small = company[
    ["证券代码", "行业名称", "省份", "城市", "上市市场", "上市日期", "上市年份", "是否ST"]
]

company_small.head()

,证券代码,行业名称,省份,城市,上市市场,上市日期,上市年份,是否ST
0,000001,货币金融服务,广东省,深圳市,SZSE,1991-04-03,1991,False
1,000002,房地产业,广东省,深圳市,SZSE,1991-01-29,1991,False
2,000004,软件和信息技术服务业,广东省,深圳市,SZSE,1991-01-14,1991,False
3,000006,房地产业,广东省,深圳市,SZSE,1992-04-27,1992,False
4,000007,房地产业,广东省,深圳市,SZSE,1992-04-13,1992,True


用 `merge()` 按证券代码合并。合并后检查行数和关键列缺失，确认匹配结果符合预期。

In [28]:
rank_with_info = finance_2020_rank_code.merge(company_small, on="证券代码", how="left")

print("合并前行数：", len(finance_2020_rank_code))
print("合并后行数：", len(rank_with_info))
print("行业缺失数量：", rank_with_info["行业名称"].isna().sum())

rank_with_info[[
    "证券代码", "证券简称", "行业名称", "省份", "营业收入_亿元", "净利率", "上市年份"
]].head(10)

合并前行数： 28
合并后行数： 28
行业缺失数量： 0


,证券代码,证券简称,行业名称,省份,营业收入_亿元,净利率,上市年份
0,000002,万科A,房地产业,广东省,4191.1200,0.1415,1991
1,000333,美的集团,电气机械及器材制造业,广东省,2857.1000,0.0963,2013
2,000338,潍柴动力,汽车制造业,山东省,1974.9100,0.0571,2004
3,600000,浦发银行,货币金融服务,上海市,1746.8700,0.3377,1999
4,600016,民生银行,货币金融服务,北京市,1667.7700,0.2105,2000
5,000001,平安银行,货币金融服务,广东省,1432.4200,0.2020,1991
6,600015,华夏银行,货币金融服务,北京市,927.1700,0.2326,2003
7,000016,深康佳A,计算机、通信和其他电子设备制造业,广东省,503.5200,0.0107,1992
8,000550,江铃汽车,汽车制造业,江西省,330.9600,0.0166,1993
9,000538,云南白药,医药制造业,云南省,327.4300,0.1683,1993


In [29]:
rank_with_info[[
    "证券代码", "证券简称", "行业名称", "省份", "营业收入_亿元", "净利率", "上市年份"
]].tail(5)

,证券代码,证券简称,行业名称,省份,营业收入_亿元,净利率,上市年份
23,000045,深纺织A,计算机、通信和其他电子设备制造业,广东省,21.0900,0.0204,1994
24,000020,深华发A,计算机、通信和其他电子设备制造业,广东省,6.9200,0.0101,1992
25,000004,国华网安,软件和信息技术服务业,广东省,2.8100,0.2206,1991
26,000638,*ST 万方,软件和信息技术服务业,吉林省,1.1100,-0.1802,1996
27,000007,*ST 全新,房地产业,广东省,0.4500,-2.7333,1992


两个表的连接键列名不同时，可以用 `left_on` 和 `right_on`。下面做一个简短演示。

In [30]:
company_key_demo = company_small.rename(columns={"证券代码": "公司代码"})

demo_merge = finance_2020_rank_code.merge(
    company_key_demo,
    left_on="证券代码",
    right_on="公司代码",
    how="left",
)

demo_merge[["证券代码", "公司代码", "行业名称"]].head()

,证券代码,公司代码,行业名称
0,000002,000002,房地产业
1,000333,000333,电气机械及器材制造业
2,000338,000338,汽车制造业
3,600000,600000,货币金融服务
4,600016,600016,货币金融服务


阶段 2 小结：我们得到了 `rank_with_info`。这一阶段带出了证券代码处理、字符串方法、日期方法、`value_counts()`、`merge()` 和合并检查。

完成本阶段后，请做最后“练习”中的阶段 2 练习。

## 阶段 3：处理真实数据中的常见小问题

目标：理解现实数据进入计算前通常需要检查，并掌握最常见的处理方法。

为了集中演示，这里从干净财务表复制出一份练习用的小表。

In [31]:
dirty = finance_code.head(12).copy()
dirty[["总资产_亿元", "净利润_亿元"]] = dirty[["总资产_亿元", "净利润_亿元"]].astype("object")

dirty.loc[1, "营业收入_亿元"] = np.nan
dirty.loc[2, "净利润_亿元"] = "--"
dirty.loc[3, "总资产_亿元"] = "15,285.79"
dirty.loc[4, "负债水平"] = ""
dirty = pd.concat([dirty, dirty.iloc[[0]]], ignore_index=True)

dirty

,证券代码,证券简称,年份,总资产_亿元,营业收入_亿元,净利润_亿元,资产负债率,净利率,资产收益率,是否盈利,负债水平
0,000001,平安银行,2018,34185.9200,1062.1200,248.1800,0.9298,0.2337,0.0073,True,较高
1,000001,平安银行,2019,39390.7000,NaN,281.9500,0.9205,0.2223,0.0072,True,较高
2,000001,平安银行,2020,44685.1400,1432.4200,--,0.9185,0.2020,0.0065,True,较高
3,000002,万科A,2018,"15,285.79",2976.7900,492.7200,0.8459,0.1655,0.0322,True,较高
4,000002,万科A,2019,17299.2900,3678.9400,551.3200,0.8436,0.1499,0.0319,True,
5,000002,万科A,2020,18691.7700,4191.1200,592.9800,0.8128,0.1415,0.0317,True,较高
6,000004,国华网安,2018,3.5100,3.6700,-0.2200,0.4780,-0.0599,-0.0627,False,正常
7,000004,国华网安,2019,14.9400,1.0800,-0.0400,0.0602,-0.0370,-0.0027,False,正常
8,000004,国华网安,2020,15.6400,2.8100,0.6200,0.0683,0.2206,0.0396,True,正常
9,000006,深振业A,2018,135.3700,25.1200,9.2300,0.5282,0.3674,0.0682,True,正常


检查缺失值和重复行。

In [32]:
print("缺失值：")
print(dirty.isna().sum())

print("重复行数量：", dirty.duplicated().sum())

缺失值：
证券代码       0
证券简称       0
年份         0
总资产_亿元     0
营业收入_亿元    1
净利润_亿元     0
资产负债率      0
净利率        0
资产收益率      0
是否盈利       0
负债水平       0
dtype: int64
重复行数量： 1


删除重复行。

In [33]:
dirty = dirty.drop_duplicates()
dirty = dirty.copy()
print("删除重复后行数：", len(dirty))

删除重复后行数： 12


把特殊文本和带逗号的数字清理成真正的数值。

In [34]:
dirty["总资产_亿元"] = (
    dirty["总资产_亿元"]
    .astype(str)
    .str.replace(",", "", regex=False)
)
dirty["总资产_亿元"] = pd.to_numeric(dirty["总资产_亿元"], errors="coerce")

dirty["净利润_亿元"] = dirty["净利润_亿元"].mask(dirty["净利润_亿元"] == "--", np.nan)
dirty["净利润_亿元"] = pd.to_numeric(dirty["净利润_亿元"], errors="coerce")

dirty[["总资产_亿元", "营业收入_亿元", "净利润_亿元"]].head()

,总资产_亿元,营业收入_亿元,净利润_亿元
0,34185.9200,1062.1200,248.1800
1,39390.7000,NaN,281.9500
2,44685.1400,1432.4200,NaN
3,15285.7900,2976.7900,492.7200
4,17299.2900,3678.9400,551.3200


缺失值可以删除，也可以填补。怎么处理取决于分析目的。这里演示两种常见做法。

In [35]:
drop_missing = dirty.dropna(subset=["营业收入_亿元", "净利润_亿元"])

fill_missing = dirty.copy()
fill_missing["营业收入_亿元"] = fill_missing["营业收入_亿元"].fillna(
    fill_missing["营业收入_亿元"].median()
)
fill_missing["净利润_亿元"] = fill_missing["净利润_亿元"].fillna(0)
fill_missing["负债水平"] = fill_missing["负债水平"].replace({"": "未知"})

fill_missing.isna().sum()

证券代码       0
证券简称       0
年份         0
总资产_亿元     0
营业收入_亿元    0
净利润_亿元     0
资产负债率      0
净利率        0
资产收益率      0
是否盈利       0
负债水平       0
dtype: int64

`map()` 适合把一组取值映射成另一组取值。

In [36]:
debt_map = {"正常": "低风险", "较高": "需关注", "未知": "待确认"}
fill_missing["负债风险"] = fill_missing["负债水平"].map(debt_map)

fill_missing[["证券简称", "负债水平", "负债风险"]].head()

,证券简称,负债水平,负债风险
0,平安银行,较高,需关注
1,平安银行,较高,需关注
2,平安银行,较高,需关注
3,万科A,较高,需关注
4,万科A,未知,待确认


`replace()` 适合替换特殊值。现实数据里常见 `999`、`-1`、`--` 等特殊编码。

In [37]:
special = pd.Series([1, 2, 999, -1, 5], name="原始值")
special.replace({999: np.nan, -1: 0})

0   1.0000
1   2.0000
2      NaN
3   0.0000
4   5.0000
Name: 原始值, dtype: float64

阶段 3 小结：这一阶段带出了 `isna()`、`dropna()`、`fillna()`、`duplicated()`、`drop_duplicates()`、`replace()`、`pd.to_numeric()`、`map()` 和清洗副本的做法。

完成本阶段后，请做最后“练习”中的阶段 3 练习。

### 副本、视图和链式赋值

直接修改数据时，建议优先使用：

```python
df.loc[条件, 列名] = 新值
```

筛选出一部分数据再修改时，显式使用 `.copy()` 生成工作副本。这样更容易判断后续操作影响的是哪一张表。

## 阶段 4：按行业和年份做汇总分析

目标：从公司层面的明细表，上升到行业和年份层面的比较。

In [38]:
analysis_df = finance_code.merge(company_small, on="证券代码", how="left")
analysis_df.head()

,证券代码,证券简称,年份,总资产_亿元,营业收入_亿元,净利润_亿元,资产负债率,净利率,资产收益率,是否盈利,负债水平,行业名称,省份,城市,上市市场,上市日期,上市年份,是否ST
0,000001,平安银行,2018,34185.9200,1062.1200,248.1800,0.9298,0.2337,0.0073,True,较高,货币金融服务,广东省,深圳市,SZSE,1991-04-03,1991,False
1,000001,平安银行,2019,39390.7000,1268.1400,281.9500,0.9205,0.2223,0.0072,True,较高,货币金融服务,广东省,深圳市,SZSE,1991-04-03,1991,False
2,000001,平安银行,2020,44685.1400,1432.4200,289.2800,0.9185,0.2020,0.0065,True,较高,货币金融服务,广东省,深圳市,SZSE,1991-04-03,1991,False
3,000002,万科A,2018,15285.7900,2976.7900,492.7200,0.8459,0.1655,0.0322,True,较高,房地产业,广东省,深圳市,SZSE,1991-01-29,1991,False
4,000002,万科A,2019,17299.2900,3678.9400,551.3200,0.8436,0.1499,0.0319,True,较高,房地产业,广东省,深圳市,SZSE,1991-01-29,1991,False


做一个普通的行业汇总。

In [39]:
industry_2020 = (
    analysis_df[analysis_df["年份"] == 2020]
    .groupby("行业名称")
    .agg(
        公司数=("证券简称", "nunique"),
        平均营业收入_亿元=("营业收入_亿元", "mean"),
        营业收入合计_亿元=("营业收入_亿元", "sum"),
        平均净利率=("净利率", "mean"),
        平均资产负债率=("资产负债率", "mean"),
    )
    .sort_values("营业收入合计_亿元", ascending=False)
)

industry_2020

,公司数,平均营业收入_亿元,营业收入合计_亿元,平均净利率,平均资产负债率
行业名称,,,,,
货币金融服务,4,1443.5575,5774.2300,0.2457,0.9191
房地产业,4,1065.4900,4261.9600,-0.5265,0.7023
电气机械及器材制造业,4,813.0450,3252.1800,0.0354,0.6761
汽车制造业,4,631.4550,2525.8200,0.0491,0.5425
计算机、通信和其他电子设备制造业,4,170.3000,681.2000,0.0262,0.5237
医药制造业,4,124.9825,499.9300,0.1034,0.3380
软件和信息技术服务业,4,52.4050,209.6200,0.0237,0.4648


也可以同时按行业和年份分组。

In [40]:
industry_year = (
    analysis_df
    .groupby(["行业名称", "年份"])
    .agg(
        公司数=("证券简称", "nunique"),
        平均营业收入_亿元=("营业收入_亿元", "mean"),
        平均净利率=("净利率", "mean"),
    )
    .reset_index()
)

industry_year.head(12)

,行业名称,年份,公司数,平均营业收入_亿元,平均净利率
0,医药制造业,2018,4,114.8000,0.1403
1,医药制造业,2019,4,113.1175,0.0428
2,医药制造业,2020,4,124.9825,0.1034
3,房地产业,2018,4,757.5500,-0.9802
4,房地产业,2019,4,939.0725,0.2717
5,房地产业,2020,4,1065.4900,-0.5265
6,汽车制造业,2018,4,516.8000,0.0634
7,汽车制造业,2019,4,560.4500,0.0532
8,汽车制造业,2020,4,631.4550,0.0491
9,电气机械及器材制造业,2018,4,755.6625,0.0425


`agg()` 可以使用自定义函数。

In [41]:
def value_range(x):
    return x.max() - x.min()

analysis_df.groupby("行业名称").agg(
    收入均值=("营业收入_亿元", "mean"),
    收入差距=("营业收入_亿元", value_range),
).head()

,收入均值,收入差距
行业名称,,
医药制造业,117.6333,297.8400
房地产业,920.7042,4190.7000
汽车制造业,569.5683,1896.3800
电气机械及器材制造业,788.7633,2810.5400
计算机、通信和其他电子设备制造业,169.5333,544.8200


每组取前几名的常用做法是：排序、分组、再 `head()`。

In [42]:
top2_by_industry = (
    analysis_df[analysis_df["年份"] == 2020]
    .sort_values(["行业名称", "营业收入_亿元"], ascending=[True, False])
    .groupby("行业名称")
    .head(2)
)

top2_by_industry[["行业名称", "证券简称", "营业收入_亿元"]].head(14)

,行业名称,证券简称,营业收入_亿元
62,医药制造业,云南白药,327.4300
56,医药制造业,丽珠集团,105.2000
5,房地产业,万科A,4191.1200
17,房地产业,深物业A,41.0400
50,汽车制造业,潍柴动力,1974.9100
65,汽车制造业,江铃汽车,330.9600
47,电气机械及器材制造业,美的集团,2857.1000
35,电气机械及器材制造业,德赛电池,193.9800
20,计算机、通信和其他电子设备制造业,深康佳A,503.5200
26,计算机、通信和其他电子设备制造业,深科技,149.6700


分组循环适合处理每组内部较复杂的逻辑。循环得到的结果可以用 `concat()` 合并回来。

In [43]:
top_list = []

for industry, group in analysis_df[analysis_df["年份"] == 2020].groupby("行业名称"):
    top_company = group.sort_values("营业收入_亿元", ascending=False).head(1)
    top_list.append(top_company)

industry_top_company = pd.concat(top_list)[
    ["行业名称", "证券代码", "证券简称", "营业收入_亿元", "净利率"]
].sort_values("营业收入_亿元", ascending=False)

industry_top_company

,行业名称,证券代码,证券简称,营业收入_亿元,净利率
5,房地产业,000002,万科A,4191.1200,0.1415
47,电气机械及器材制造业,000333,美的集团,2857.1000,0.0963
50,汽车制造业,000338,潍柴动力,1974.9100,0.0571
77,货币金融服务,600000,浦发银行,1746.8700,0.3377
20,计算机、通信和其他电子设备制造业,000016,深康佳A,503.5200,0.0107
62,医药制造业,000538,云南白药,327.4300,0.1683
68,软件和信息技术服务业,000555,神州信息,106.8600,0.0436


`concat()` 还可以横向或纵向拼接表。横向拼接时，它会按索引对齐。和按当前显示顺序直接粘贴相比，按索引对齐更适合保留行标签的含义。

In [44]:
left = pd.Series(["A", "B", "C"], index=[1, 2, 3], name="name")
right = pd.Series([90, 80, 70], index=[3, 2, 1], name="score")

pd.concat([left, right], axis=1)

,name,score
1,A,70
2,B,80
3,C,90


按当前行顺序拼接时，可以先重置索引。

In [45]:
pd.concat(
    [left.reset_index(drop=True), right.reset_index(drop=True)],
    axis=1,
)

,name,score
0,A,90
1,B,80
2,C,70


阶段 4 小结：这一阶段带出了 `groupby()`、多指标 `agg()`、自定义聚合、每组取前几名、分组循环和 `concat()`。

完成本阶段后，请做最后“练习”中的阶段 4 练习。

## 阶段 5：重建更适合分析的数据表

目标：把明细表改造成更适合回答问题的表。分析时可以根据问题重新组织数据形状。

把公司年度营业收入整理成宽表，每家公司一行，每一年一列。

In [46]:
revenue_wide = analysis_df.pivot_table(
    index=["证券代码", "证券简称", "行业名称"],
    columns="年份",
    values="营业收入_亿元",
)

revenue_wide.head()

,,年份,2018,2019,2020
证券代码,证券简称,行业名称,,,
000001,平安银行,货币金融服务,1062.1200,1268.1400,1432.4200
000002,万科A,房地产业,2976.7900,3678.9400,4191.1200
000004,国华网安,软件和信息技术服务业,3.6700,1.0800,2.8100
000006,深振业A,房地产业,25.1200,37.3100,29.3500
000007,*ST 全新,房地产业,0.4200,0.4200,0.4500


In [47]:
revenue_wide["收入增长率_2018_2020"] = revenue_wide[2020] / revenue_wide[2018] - 1

company_summary = (
    revenue_wide
    .reset_index()
    .sort_values("收入增长率_2018_2020", ascending=False)
)

company_summary.head(10)

年份,证券代码,证券简称,行业名称,2018,2019,2020,收入增长率_2018_2020
10,000045,深纺织A,计算机、通信和其他电子设备制造业,12.7200,21.5800,21.0900,0.6580
5,000011,深物业A,房地产业,27.8700,39.6200,41.0400,0.4726
9,000030,富奥股份,汽车制造业,78.5300,100.6400,111.1300,0.4151
1,000002,万科A,房地产业,2976.7900,3678.9400,4191.1200,0.4079
0,000001,平安银行,货币金融服务,1062.1200,1268.1400,1432.4200,0.3486
26,600015,华夏银行,货币金融服务,694.0300,827.6900,927.1700,0.3359
27,600016,民生银行,货币金融服务,1288.9300,1549.8200,1667.7700,0.2939
16,000338,潍柴动力,汽车制造业,1592.5600,1743.6100,1974.9100,0.2401
20,000538,云南白药,医药制造业,267.0800,296.6500,327.4300,0.2260
18,000513,丽珠集团,医药制造业,88.6100,93.8500,105.2000,0.1872


`set_index()` 和 `reset_index()` 常用于在“普通列”和“索引”之间切换。

In [48]:
indexed = company_summary.set_index("证券代码")
indexed.head()

年份,证券简称,行业名称,2018,2019,2020,收入增长率_2018_2020
证券代码,,,,,,
000045,深纺织A,计算机、通信和其他电子设备制造业,12.7200,21.5800,21.0900,0.6580
000011,深物业A,房地产业,27.8700,39.6200,41.0400,0.4726
000030,富奥股份,汽车制造业,78.5300,100.6400,111.1300,0.4151
000002,万科A,房地产业,2976.7900,3678.9400,4191.1200,0.4079
000001,平安银行,货币金融服务,1062.1200,1268.1400,1432.4200,0.3486


In [49]:
indexed.reset_index().head()

年份,证券代码,证券简称,行业名称,2018,2019,2020,收入增长率_2018_2020
0,000045,深纺织A,计算机、通信和其他电子设备制造业,12.7200,21.5800,21.0900,0.6580
1,000011,深物业A,房地产业,27.8700,39.6200,41.0400,0.4726
2,000030,富奥股份,汽车制造业,78.5300,100.6400,111.1300,0.4151
3,000002,万科A,房地产业,2976.7900,3678.9400,4191.1200,0.4079
4,000001,平安银行,货币金融服务,1062.1200,1268.1400,1432.4200,0.3486


也可以自己构造小的 `Series` 和 `DataFrame`，理解 pandas 对象如何组成表格。

In [50]:
s = pd.Series([10, 20, 30], index=["A", "B", "C"], name="得分")
demo_df = pd.DataFrame({
    "公司": ["甲", "乙", "丙"],
    "收入": [100, 120, 80],
})

print(s)
demo_df

A    10
B    20
C    30
Name: 得分, dtype: int64


,公司,收入
0,甲,100
1,乙,120
2,丙,80


保存结果。Excel 适合给人看，CSV 更通用。默认情况下，pandas 会把索引也保存成文件中的一列。普通表格通常使用 `index=False`。索引本身有意义时，可以先用 `reset_index()` 把它变成普通列，再保存。

In [51]:
finance_2020_rank_code.to_excel("data/finance_2020_rank_full.xlsx", index=False)
industry_2020.to_excel("data/industry_2020_full_summary.xlsx")
company_summary.to_excel("data/company_full_summary.xlsx", index=False)
company_summary.to_csv("data/company_full_summary.csv", index=False)

print("已保存阶段成果")

已保存阶段成果


In [52]:
pd.read_csv("data/company_full_summary.csv").head()

,证券代码,证券简称,行业名称,2018,2019,2020,收入增长率_2018_2020
0,45,深纺织A,计算机、通信和其他电子设备制造业,12.7200,21.5800,21.0900,0.6580
1,11,深物业A,房地产业,27.8700,39.6200,41.0400,0.4726
2,30,富奥股份,汽车制造业,78.5300,100.6400,111.1300,0.4151
3,2,万科A,房地产业,2976.7900,3678.9400,4191.1200,0.4079
4,1,平安银行,货币金融服务,1062.1200,1268.1400,1432.4200,0.3486


阶段 5 小结：这一阶段带出了 `pivot_table()`、`set_index()`、`reset_index()`、构造 `Series` / `DataFrame`，以及保存 Excel 和 CSV。

完成本阶段后，请做最后“练习”中的阶段 5 练习。

## 阶段 6：时间序列入门

目标：理解带日期的数据如何切片、滞后、差分、计算增长率和重采样。这里构造一份日度价格数据。

In [53]:
dates = pd.date_range("2020-01-01", "2020-03-31", freq="B")
rng = np.random.default_rng(42)

returns = pd.DataFrame(
    {
        "平安银行": rng.normal(0.0005, 0.015, len(dates)),
        "万科A": rng.normal(0.0003, 0.018, len(dates)),
    },
    index=dates,
)

prices = 100 * (1 + returns).cumprod()
prices.head()

,平安银行,万科A
2020-01-01,100.5071,101.4580
2020-01-02,98.9894,100.8516
2020-01-03,100.1532,100.0425
2020-01-06,101.6163,101.6176
2020-01-07,98.6933,101.2981


日期作为索引后，可以直接按日期字符串切片。

In [54]:
prices.loc["2020-02"].head()

,平安银行,万科A
2020-02-03,100.3306,95.3827
2020-02-04,99.7361,96.1780
2020-02-05,99.2592,97.3587
2020-02-06,100.1014,97.2153
2020-02-07,100.7001,96.5038


In [55]:
prices.loc["2020-02-10":"2020-02-20"]

,平安银行,万科A
2020-02-10,101.3739,96.3943
2020-02-11,102.0797,93.4955
2020-02-12,105.4100,91.0882
2020-02-13,104.8201,88.9468
2020-02-14,104.0671,87.3769
2020-02-17,102.8489,88.0318
2020-02-18,103.8506,86.6234
2020-02-19,105.6612,86.0598
2020-02-20,105.5334,88.0982


`shift()` 做滞后，`diff()` 做差分，`pct_change()` 计算百分比变化。

In [56]:
ts = prices["平安银行"].to_frame("price")
ts["lag_price"] = ts["price"].shift(1)
ts["diff"] = ts["price"].diff()
ts["return"] = ts["price"].pct_change()

ts.head()

,price,lag_price,diff,return
2020-01-01,100.5071,NaN,NaN,NaN
2020-01-02,98.9894,100.5071,-1.5176,-0.0151
2020-01-03,100.1532,98.9894,1.1638,0.0118
2020-01-06,101.6163,100.1532,1.4631,0.0146
2020-01-07,98.6933,101.6163,-2.9230,-0.0288


如果已经有收益率，也可以用 `cumprod()` 还原价格路径。

In [57]:
rebuilt_price = 100 * (1 + ts["return"].fillna(0)).cumprod()
rebuilt_price.head()

2020-01-01   100.0000
2020-01-02    98.4900
2020-01-03    99.6479
2020-01-06   101.1037
2020-01-07    98.1954
Freq: B, Name: return, dtype: float64

`resample()` 可以把高频数据汇总到较低频率。例如把日度价格变成月末价格，把日收益变成月收益。

In [58]:
month_end_price = prices.resample("ME").last()
month_return = month_end_price.pct_change()

month_end_price

,平安银行,万科A
2020-01-31,100.5133,94.5363
2020-02-29,105.1916,85.0293
2020-03-31,108.1457,84.6394


In [59]:
month_return

,平安银行,万科A
2020-01-31,NaN,NaN
2020-02-29,0.0465,-0.1006
2020-03-31,0.0281,-0.0046


阶段 6 小结：这一阶段带出了 `date_range()`、日期索引、按日期切片、`shift()`、`diff()`、`pct_change()`、`cumprod()` 和 `resample()`。

完成本阶段后，请做最后“练习”中的阶段 6 练习。

## 练习

下面的练习继续使用本章两张表。建议每完成一步都看一眼结果，例如使用 `head()`、`tail()`、`shape` 或者简单统计。题目中的变量名是建议变量名，便于课堂核对。

### 阶段 1 练习：从一张财务表得到公司排名

**练习 1.1：找出低负债且盈利的公司**

读取 `finance_teaching_clean.xlsx`，赋值给变量 `finance_ex1`。暂时删除 `证券代码` 列。筛选 2020 年同时满足以下条件的公司：`净利润_亿元 > 0`、`资产负债率 < 0.6`。结果赋值给变量 `low_debt_profit_2020`，只保留 `证券简称`、`营业收入_亿元`、`净利润_亿元`、`资产负债率` 四列，并按 `资产负债率` 从低到高排序。打印结果行数，并显示前 5 行。

**练习 1.2：找出收入高但净利率较低的公司**

在 `finance_ex1` 中新增 `净利率 = 净利润_亿元 / 营业收入_亿元`。筛选 2020 年 `营业收入_亿元` 高于当年中位数、同时 `净利率` 低于当年中位数的公司，赋值给变量 `high_revenue_low_margin`。这个结果代表“规模不小、利润率相对偏低”的公司。显示 `证券简称`、`营业收入_亿元`、`净利润_亿元`、`净利率`，并查看前 5 行和后 5 行。

**练习 1.3：给公司打上资产规模标签**

用 `pd.cut()` 根据 `总资产_亿元` 生成 `资产规模`：`0-100` 为“小”，`100-1000` 为“中”，`1000` 以上为“大”。统计 2020 年不同 `资产规模` 的公司数量，赋值给变量 `size_counts_2020`。要求输出计数结果。

### 阶段 2 练习：合并公司信息并解释排名

**练习 2.1：处理证券代码并合并行业信息**

重新读取 `finance_teaching_clean.xlsx` 和 `company_profile_teaching_clean.xlsx`。读取时把 `证券代码` 处理成 6 位字符串。筛选 2020 年财务数据，与公司信息表按 `证券代码` 合并，赋值给变量 `finance_company_2020`。检查合并前后行数是否一致，并检查 `行业名称` 是否有缺失。

**练习 2.2：找出制造业中收入最高的公司**

在 `finance_company_2020` 中筛选 `行业名称` 包含“制造”的公司，赋值给变量 `manufacturing_2020`。按 `营业收入_亿元` 从高到低排序，显示前 8 行。结果应包含 `证券代码`、`证券简称`、`行业名称`、`省份`、`营业收入_亿元`、`净利润_亿元`。

**练习 2.3：比较早上市公司和较晚上市公司**

把公司表中的 `上市日期` 转为日期格式，并生成 `上市年份`。在 `finance_company_2020` 中新增 `上市阶段`：`上市年份 < 2000` 标记为“较早上市”，否则标记为“较晚上市”。分别计算两组公司的平均 `营业收入_亿元` 和平均 `净利率`。

### 阶段 3 练习：处理真实数据中的小问题

**练习 3.1：清洗一张带问题的小表**

从带代码的财务表中取前 12 行，复制为 `dirty_ex1`。为练习清洗操作，加入以下情况：把第 2 行 `营业收入_亿元` 改成缺失值；把第 3 行 `净利润_亿元` 改成 `"--"`；把第 4 行 `总资产_亿元` 改成带逗号的字符串；再重复添加第 1 行。完成缺失值检查、重复行检查、去重、数值转换。最终结果赋值给变量 `dirty_ex1_clean`，要求 `总资产_亿元` 和 `净利润_亿元` 都是数值列。

**练习 3.2：替换特殊值并填补缺失**

在 `dirty_ex1_clean` 中，把 `净利润_亿元` 的缺失值填为 0，把 `营业收入_亿元` 的缺失值填为该列中位数。新增 `是否盈利` 列：`净利润_亿元 > 0` 为 `True`，否则为 `False`。输出每列缺失值数量，并显示 `证券简称`、`营业收入_亿元`、`净利润_亿元`、`是否盈利`。

**练习 3.3：用映射生成风险标签**

根据 `资产负债率` 新增 `负债水平`：大于等于 0.7 为“较高”，否则为“正常”。再用 `map()` 把“较高”映射为“需关注”，把“正常”映射为“低风险”，生成 `负债风险`。统计不同 `负债风险` 的数量。

### 阶段 4 练习：按行业和年份汇总

**练习 4.1：行业年度摘要表**

把财务表和公司信息表合并为 `analysis_ex4`。按 `行业名称` 和 `年份` 分组，计算 `公司数`、`营业收入合计_亿元`、`平均净利率`、`平均资产负债率`，赋值给变量 `industry_year_summary`。显示前 12 行。

**练习 4.2：每个行业找两家公司**

在 2020 年数据中，按 `行业名称` 分组，找出每个行业 `营业收入_亿元` 最高的 2 家公司，赋值给变量 `top2_revenue_by_industry`。结果只保留 `行业名称`、`证券代码`、`证券简称`、`营业收入_亿元`、`净利率`，并按行业名称和营业收入排序。

**练习 4.3：计算行业内部收入差距**

自定义函数 `value_range(x)`，返回 `x.max() - x.min()`。按 `行业名称` 分组，计算 2020 年各行业营业收入的均值和收入差距，赋值给变量 `industry_gap_2020`。按收入差距从高到低排序。

### 阶段 5 练习：重建分析表

**练习 5.1：构造公司收入宽表**

用 `pivot_table()` 把 `analysis_ex4` 整理为每家公司一行、年份为列、值为 `营业收入_亿元` 的宽表，赋值给变量 `revenue_wide_ex5`。计算 `收入增长率_2018_2020 = 2020 / 2018 - 1`，按增长率从高到低排序，显示前 10 行。

**练习 5.2：构造公司综合摘要表**

基于 `analysis_ex4` 构造 `company_profile_summary`，每家公司一行，至少包含：`证券代码`、`证券简称`、`行业名称`、`省份`、`2018` 年营业收入、`2020` 年营业收入、`收入增长率_2018_2020`、`2020` 年资产负债率。按 `收入增长率_2018_2020` 从高到低排序，显示前 10 行。

**练习 5.3：核对公司摘要表**

检查 `company_profile_summary` 的行数是否等于公司数量。再按 `行业名称` 统计公司数量，并显示统计结果。最后显示 `company_profile_summary` 的前 5 行和后 5 行。

### 阶段 6 练习：时间序列入门

**练习 6.1：构造价格序列并计算收益率**

构造 2021 年 1 月到 2021 年 6 月的工作日日期。使用随机数生成一列日收益率 `return`，从初始价格 100 出发构造价格 `price`。把结果赋值给以日期为索引的变量 `price_df`，并新增 `lag_price`、`diff`、`pct_return` 三列。显示前 8 行。

**练习 6.2：按月份观察价格变化**

从 `price_df` 中筛选 2021 年 3 月的数据，赋值给变量 `march_price`。显示前 5 行和后 5 行。然后用 `resample("ME").last()` 得到月末价格，赋值给变量 `month_end_price_ex6`。

**练习 6.3：从月末价格计算月收益**

基于 `month_end_price_ex6` 计算月收益率，赋值给变量 `month_return_ex6`。找出月收益率最高的月份和最低的月份，并打印对应月份和收益率。